In [1]:
import sys, os
from pathlib import Path
import matplotlib.pyplot as plt

from shutil import rmtree
import vtk
import numpy as np
import matplotlib.pyplot as plt

root_path = os.path.join(Path().resolve().parent.parent)
package_path = os.path.join(root_path, "hamageolib")
test_fixture_path = os.path.join(root_path, "big_tests", "TwoDSubduction", "eba_cdpt_coh500_SA80.0_cd7.5_log_ss300.0")
result_path = os.path.join(Path().resolve(), "results")

if str(package_path) not in sys.path:
    sys.path.insert(0, str(package_path))

if not os.path.isdir(result_path):
    os.mkdir(result_path)

In [2]:
case_path = os.path.join(Path().resolve(), "../../big_tests/TwoDSubduction/eba_cdpt_coh500_SA80.0_cd7.5_log_ss300.0")

assert(os.path.isdir(case_path))

file_path = os.path.join(case_path, "output", "solution", "solution-00094.pvtu")

assert(os.path.isfile(file_path))

In [3]:
# Read the PVTU file as an Unstructured Grid
reader = vtk.vtkXMLPUnstructuredGridReader()
reader.SetFileName(file_path)
reader.Update()
u_grid = reader.GetOutput()

# Convert Unstructured Grid to PolyData
geometry_filter = vtk.vtkGeometryFilter()
geometry_filter.SetInputData(reader.GetOutput())
geometry_filter.Update()
poly_data = geometry_filter.GetOutput()

In [ ]:
from utils.vtk_utilities import estimate_memory_usage

# Get the number of points and cells in u_grid (full unstructured grid)
num_points_u = u_grid.GetNumberOfPoints()
num_cells_u = u_grid.GetNumberOfCells()

# Get the number of points and cells in poly_data (surface representation)
num_points_poly = poly_data.GetNumberOfPoints()
num_cells_poly = poly_data.GetNumberOfCells()

# Print results
print("Unstructured Grid (Full Volume Mesh):")
print(f"  - Number of Points: {num_points_u}")
print(f"  - Number of Cells: {num_cells_u}")

print("\nPolyData (Extracted Surface Mesh):")
print(f"  - Number of Points: {num_points_poly}")
print(f"  - Number of Cells: {num_cells_poly}")

# Example usage
# Assuming u_grid is already loaded
memory_usage_mb = estimate_memory_usage(u_grid)
print(f"\nEstimated Memory Usage: {memory_usage_mb:.2f} MB")

In [ ]:
import vtk

def extract_slab(u_grid, field_name, threshold_value=0.5):
    """
    Extracts the slab shape based on a composition marker in a vtkUnstructuredGrid.

    Parameters:
    u_grid (vtkUnstructuredGrid): Input unstructured grid.
    field_name (str): Name of the composition field (default: "sp_crust").
    threshold_value (float): Value used to filter slab points (default: 0.5).

    Returns:
    vtkPolyData: Extracted surface of the slab.
    """
    # Check if the field exists
    point_data = u_grid.GetPointData()
    scalar_field = point_data.GetArray(field_name)
    
    if not scalar_field:
        print(f"Error: Field '{field_name}' not found in point data.")
        return None

    # Threshold filter to extract slab region
    threshold_filter = vtk.vtkThreshold()
    threshold_filter.SetInputData(u_grid)
    threshold_filter.SetInputArrayToProcess(0, 0, 0, 0, field_name)  # Use scalar field
    threshold_filter.ThresholdByUpper(threshold_value)  # Keep values >= threshold
    threshold_filter.Update()

    # Convert to surface representation
    geometry_filter = vtk.vtkGeometryFilter()
    geometry_filter.SetInputData(threshold_filter.GetOutput())
    geometry_filter.Update()

    return geometry_filter.GetOutput()  # PolyData containing the slab surface

# Example Usage
slab_polydata = extract_slab(u_grid, "spcrust")

if slab_polydata:
    print(f"Extracted Slab Shape: {slab_polydata.GetNumberOfPoints()} points, {slab_polydata.GetNumberOfCells()} cells.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define zoom-in region (adjust as needed)
x_range = (4.8e6, 5.2e6)  # Example: X range to zoom in
y_range = (3.5e6, 4.0e6)  # Example: Y range to zoom in

# Extract points from slab_polydata
points = slab_polydata.GetPoints()
num_points = slab_polydata.GetNumberOfPoints()

# Convert VTK points to NumPy array
points_array = np.array([points.GetPoint(i) for i in range(num_points)])

# Extract X and Y coordinates
x, y = points_array[:, 0], points_array[:, 1]

# Create a mask for points inside the zoomed-in region
zoom_mask = (x >= x_range[0]) & (x <= x_range[1]) & (y >= y_range[0]) & (y <= y_range[1])
zoomed_points = points_array[zoom_mask]

# Plotting
fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# Full slab plot
axs[0].scatter(x, y, s=1, color='black', label="All Points")
axs[0].set_xlabel("X Coordinate")
axs[0].set_ylabel("Y Coordinate")
axs[0].set_title("Full Slab Shape")
axs[0].legend()
axs[0].grid(True)

# Highlight zoomed region
axs[0].plot([x_range[0], x_range[0], x_range[1], x_range[1], x_range[0]],
            [y_range[0], y_range[1], y_range[1], y_range[0], y_range[0]],
            'r-', linewidth=1.5, label="Zoom Region")

# Zoomed-in subplot
axs[1].scatter(zoomed_points[:, 0], zoomed_points[:, 1], s=5, color='red', label="Zoomed Points")
axs[1].set_xlim(x_range)
axs[1].set_ylim(y_range)
axs[1].set_xlabel("X Coordinate (Zoomed)")
axs[1].set_ylabel("Y Coordinate (Zoomed)")
axs[1].set_title("Zoomed-In Slab Region")
axs[1].legend()
axs[1].grid(True)

plt.tight_layout()
plt.show()
